In [1]:
import sys
import os

# Obtenemos la ruta de la carpeta actual (/Workspace/.../silver)
current_dir = os.getcwd()
# Subimos un nivel hacia la carpeta raíz de tu proyecto
parent_dir = os.path.dirname(current_dir)

# Agregamos la raíz de tu proyecto al cerebro de Python
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from src.etl_master import LakehouseMaster
from src.analytics_utils import AnalyticsUtils
from pyspark.sql.functions import col

In [3]:
class StOrderItems(LakehouseMaster):
    
    def transform(self, df):
        # Deduplicación
        df_clean = df.dropDuplicates(["id"])
        
        # Casteo explícito de métricas
        df_clean = df_clean.withColumn("price", col("price").cast("decimal(10,2)")) \
                           .withColumn("quantity", col("quantity").cast("integer"))
        
        # Limpieza genérica
        df_clean = AnalyticsUtils.cleaning_df_spark(df_clean)
        
        return df_clean